In [100]:
import pandas as pd


orders = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\orders_dataset.csv"
)

customers = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\customers_dataset.csv"
)

payments = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\payments_dataset.csv"
)

# Convert purchase timestamp
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    format="mixed"
)

# Keep orders before the modeling cutoff
cutoff_date = pd.Timestamp("2018-07-01")

train_orders = orders[
    orders["order_purchase_timestamp"] < cutoff_date
].copy()
# Add customer_unique_id
train_orders = train_orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Payments:", payments.shape)
print("Training orders:", train_orders.shape)

Orders: (99441, 8)
Customers: (99441, 5)
Payments: (103886, 5)
Training orders: (81022, 9)


In [ ]:
# Create customer-level training and testing datasets using two time cutoffs

train_cutoff = pd.Timestamp("2017-10-01")
test_cutoff = pd.Timestamp("2018-01-01")

print("Training cutoff:", train_cutoff)
print("Testing cutoff:", test_cutoff)


In [ ]:
# Aggregate payment information at order level

order_payments = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
)

orders_with_payment = train_orders.merge(
    order_payments,
    on="order_id",
    how="left"
)

orders_with_payment["payment_value"] = (
    orders_with_payment["payment_value"].fillna(0)
)


In [ ]:
# Create one customer-level dataset for a given cutoff date

def create_customer_dataset(cutoff_date):

    historical = orders_with_payment[
        orders_with_payment["order_purchase_timestamp"] <= cutoff_date
    ]

    customer_data = (
        historical
        .groupby("customer_unique_id")
        .agg(
            total_orders=("order_id", "nunique"),
            total_spending=("payment_value", "sum"),
            average_order_value=("payment_value", "mean"),
            first_order_date=("order_purchase_timestamp", "min"),
            last_order_date=("order_purchase_timestamp", "max")
        )
        .reset_index()
    )

    customer_data["customer_lifetime_days"] = (
        customer_data["last_order_date"]
        - customer_data["first_order_date"]
    ).dt.days

    customer_data["order_frequency"] = (
        customer_data["total_orders"]
        / (customer_data["customer_lifetime_days"] + 1)
    )

    future_orders = train_orders[
        (train_orders["order_purchase_timestamp"] > cutoff_date) &
        (
            train_orders["order_purchase_timestamp"]
            <= cutoff_date + pd.Timedelta(days=90)
        )
    ]

    future_customers = set(
        future_orders["customer_unique_id"].unique()
    )

    customer_data["repeat_purchase"] = (
        customer_data["customer_unique_id"]
        .isin(future_customers)
        .astype(int)
    )

    return customer_data


In [124]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    print(
        f"Threshold: {threshold:} | "
        f"Precision: {precision_score(y_test, y_pred_threshold):} | "
        f"Recall: {recall_score(y_test, y_pred_threshold):} | "
        f"F1: {f1_score(y_test, y_pred_threshold):}"
    )

Threshold: 0.05 | Precision: 0.006627922452420034 | Recall: 0.9701298701298702 | F1: 0.013165895571711831
Threshold: 0.1 | Precision: 0.006715523224893408 | Recall: 0.9675324675324676 | F1: 0.01333846580787238
Threshold: 0.15 | Precision: 0.006723335653830652 | Recall: 0.9532467532467532 | F1: 0.01335249495188372
Threshold: 0.2 | Precision: 0.006792642701403254 | Recall: 0.9467532467532468 | F1: 0.013488509787958406
Threshold: 0.3 | Precision: 0.006799142801480616 | Recall: 0.9064935064935065 | F1: 0.013497051145702408
Threshold: 0.4 | Precision: 0.007013021060910601 | Recall: 0.8337662337662337 | F1: 0.013909049548280868
Threshold: 0.5 | Precision: 0.013333333333333334 | Recall: 0.2779220779220779 | F1: 0.02544589774078478


In [ ]:
# Create training and testing datasets

train_data = create_customer_dataset(train_cutoff)
test_data = create_customer_dataset(test_cutoff)

print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)

print("\nTraining target:")
print(train_data["repeat_purchase"].value_counts())

print("\nTesting target:")
print(test_data["repeat_purchase"].value_counts())


In [121]:
# Select model features

features = [
    "total_orders",
    "customer_lifetime_days",
    "order_frequency",
    "total_spending",
    "average_order_value"
]

X_train = train_data[features]
y_train = train_data["repeat_purchase"]

X_test = test_data[features]
y_test = test_data["repeat_purchase"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (297322, 5)
X_test: (117355, 5)


In [120]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model_lr.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [119]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred = model_lr.predict(X_test)
y_prob = model_lr.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9660772868646414
Precision: 0.02931691586045148
Recall: 0.12987012987012986
F1 Score: 0.04783544606553456
ROC-AUC: 0.5489724792512458

Confusion Matrix:
[[113274   3311]
 [   670    100]]


In [110]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [111]:
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
threshold = 0.50

# Convert probabilities into predictions
y_pred_rf = (y_prob_rf >= threshold).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.8603212474969111
Precision: 0.013333333333333334
Recall: 0.2779220779220779
F1 Score: 0.02544589774078478
ROC-AUC: 0.5842143043729868

Confusion Matrix:
[[100749  15836]
 [   556    214]]


In [122]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

feature_importance

,feature,importance
4,average_order_value,0.395724
3,total_spending,0.388453
2,order_frequency,0.099264
1,customer_lifetime_days,0.085526
0,total_orders,0.031033


In [114]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model_rf, "../models/repeat_purchase_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [115]:
os.path.exists("../models/repeat_purchase_model.pkl")

True

In [116]:
powerbi_data = X_test.copy()

powerbi_data["repeat_purchase"] = y_test.values
powerbi_data["predicted_repeat_purchase"] = y_pred_rf
powerbi_data["repeat_probability"] = y_prob_rf

powerbi_data.head()

,total_orders,customer_lifetime_days,order_frequency,total_spending,average_order_value,repeat_purchase,predicted_repeat_purchase,repeat_probability
297322,1,0,1.0,86.22,86.22,0,1,0.525967
297323,1,0,1.0,43.62,43.62,0,0,0.481102
297324,1,0,1.0,196.89,196.89,0,0,0.458604
297325,1,0,1.0,150.12,150.12,0,0,0.454552
297326,1,0,1.0,29.00,29.00,0,0,0.480285


In [86]:
powerbi_data.to_csv(
    "../data/processed/powerbi_customer_predictions.csv",
    index=False
)

print("Power BI dataset saved.")

Power BI dataset saved.


In [117]:
feature_importance.to_csv(
    "../data/processed/feature_importance.csv",
    index=False
)

print("Feature importance saved.")

Feature importance saved.


In [93]:
import pandas as pd


print("Payments shape:", payments.shape)
payments.head()

Payments shape: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
